# 8. Evaluación e Insights Estratégicos

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, learning_curve
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression

RANDOM_STATE = 42
DATA_DIR = "../data"
MODEL_CSV = f"{DATA_DIR}/processed/gb_videos_model.csv"
FIG_DIR = "../reports/figures"
os.makedirs(FIG_DIR, exist_ok=True)
sns.set_theme(style="whitegrid")

In [ ]:
# Reentrenamos el mejor modelo (Random Forest) para el análisis de evaluación
df = pd.read_csv(MODEL_CSV)
X = df.drop(columns="views")
y = np.log1p(df["views"])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

best_model = RandomForestRegressor(
    n_estimators=200, max_depth=15, n_jobs=-1, random_state=RANDOM_STATE
)
best_name = "Random Forest"
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

## 8.1 Análisis de residuos

In [ ]:
# Análisis de residuos
residuos = y_test - y_pred
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(residuos, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribución de residuos")
axes[1].scatter(y_pred, residuos, alpha=0.3, edgecolors="none")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_xlabel("Predicción")
axes[1].set_ylabel("Residuo")
axes[1].set_title("Residuos vs Predicción")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/model_residuals.png", dpi=300, bbox_inches="tight")
plt.show()

## 8.2 Importancia de variables

In [ ]:
# Importancia de variables (Random Forest)
importancias = pd.Series(best_model.feature_importances_, index=X.columns)
top = importancias.sort_values(ascending=False).head(15)
plt.figure(figsize=(10, 6))
sns.barplot(x=top.values, y=top.index, palette="viridis", hue=top.index, legend=False)
plt.title("Top 15 variables más importantes")
plt.xlabel("Importancia")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/model_feature_importance.png", dpi=300, bbox_inches="tight")
plt.show()

## 8.3 Curva de aprendizaje

In [ ]:
# Curva de aprendizaje para diagnosticar sesgo/varianza
train_sizes, train_scores, test_scores = learning_curve(
    best_model, X, y, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 5), random_state=RANDOM_STATE,
)
plt.figure(figsize=(10, 6))
plt.plot(train_sizes, -train_scores.mean(axis=1), "o-", label="Train")
plt.plot(train_sizes, -test_scores.mean(axis=1), "o-", label="Validación")
plt.xlabel("Tamaño de entrenamiento")
plt.ylabel("MAE")
plt.title("Curva de aprendizaje")
plt.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/model_learning_curve.png", dpi=300, bbox_inches="tight")
plt.show()

## 8.4 Diagnóstico de sesgo/varianza

In [ ]:
# Diagnóstico de sesgo/varianza
train_score = best_model.score(X_train, y_train)
test_score = best_model.score(X_test, y_test)
print(f"R2 train: {train_score:.4f} | R2 test: {test_score:.4f}")
if train_score - test_score > 0.1:
    print("Diagnóstico: posible sobreajuste (gap train-test alto).")
elif test_score < 0.6:
    print("Diagnóstico: posible subajuste (R2 test bajo).")
else:
    print("Diagnóstico: buen balance sesgo-varianza.")

## 8.5 Insights estratégicos

In [ ]:
print('''
1. Entertainment y Music dominan el volumen, pero no necesariamente el engagement.
2. El engagement_rate y likes_per_view son mejores predictores que los likes absolutos.
3. Los videos publicados en fines de semana pueden mostrar patrones diferentes de consumo.
4. Unos pocos canales concentran la mayoría de apariciones en tendencias.
5. Las regiones con más actividad también presentan mayor dispersión en vistas.
''')